In [1]:
#Import Libraries

In [2]:
import pandas as pd
import numpy as np

In [3]:
#Load Your Datasets

In [4]:
karachi = pd.read_csv("karachi_crime_dataset.csv")
print(karachi.head())

la = pd.read_csv("crime_in_la.csv")
print(la.head())

       INCIDENT_ID        TOWN TOWN_RISK_LEVEL  TOWN_PRIORITY_RANK  \
0  INCIDENT_000001  Lyari Town            High                   1   
1  INCIDENT_000002  Lyari Town            High                   1   
2  INCIDENT_000003  Lyari Town            High                   1   
3  INCIDENT_000004  Lyari Town            High                   1   
4  INCIDENT_000005  Lyari Town            High                   1   

  SUBDIVISION SUBDIVISION_RISK_LEVEL  SUBDIVISION_PRIORITY_RANK  \
0    Baghdadi                   High                          1   
1    Baghdadi                   High                          1   
2    Baghdadi                   High                          1   
3    Baghdadi                   High                          1   
4    Baghdadi                   High                          1   

   SEVERITY_SCORE SEVERITY        DATE  ...  LONGITUDE     CRIME_TYPE  \
0              10     High  2024-12-15  ...    66.9998  Gang Violence   
1              10     High  20

In [5]:
#Add City Column

In [6]:
karachi['City'] = 'Karachi'
la['City'] = 'Los Angeles'

In [7]:
#Standardize Columns

In [8]:
karachi_clean = karachi[['DATE', 'LATITUDE', 'LONGITUDE', 'CRIME_TYPE', 'City']].copy()
karachi_clean.columns = ['Date', 'Latitude', 'Longitude', 'Crime_Type', 'City']

la_clean = la[['DATE OCC', 'LAT', 'LON', 'Crm Cd Desc', 'City']].copy()
la_clean.columns = ['Date', 'Latitude', 'Longitude', 'Crime_Type', 'City']

In [9]:
#Combine Data

In [10]:
df = pd.concat([karachi_clean, la_clean], ignore_index=True)

In [11]:
#Clean Data

In [12]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Month'] = df['Date'].dt.month

df = df.dropna(subset=['Latitude', 'Longitude', 'Crime_Type'])
df = df.reset_index(drop=True)

df.head()

,Date,Latitude,Longitude,Crime_Type,City,Month
0,2024-12-15,24.8592,66.9998,Gang Violence,Karachi,12.0
1,2020-12-03,24.8592,66.9998,Murder,Karachi,12.0
2,2025-01-24,24.8592,66.9998,Murder,Karachi,1.0
3,2023-02-18,24.8592,66.9998,Gang Violence,Karachi,2.0
4,2025-04-13,24.8592,66.9998,Gang Violence,Karachi,4.0


In [13]:
#Save Final Dataset (IMPORTANT)

In [14]:
df.to_csv(r"C:\Users\yadav\OneDrive\사진\Desktop\CrimeProject\combined_crime_data.csv", index=False)

In [15]:
#MACHINE LEARNING (Light Model)

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = df[['Latitude', 'Longitude', 'Month']]
y = df['Crime_Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=50)
model.fit(X_train, y_train)

,n_estimators,50
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Take sample (avoid memory error)
df_sample = df.sample(20000, random_state=42)

# Encode
df_sample['Crime_Type'] = df_sample['Crime_Type'].astype('category').cat.codes
df_sample['City'] = df_sample['City'].astype('category').cat.codes

X = df_sample[['Latitude', 'Longitude', 'Month', 'City']]
y = df_sample['Crime_Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = DecisionTreeClassifier(max_depth=10)
model.fit(X_train, y_train)

print("Model trained successfully ✅")

Model trained successfully ✅


In [18]:
#Prediction

In [19]:
new_data = pd.DataFrame({
    'Latitude': [24.86],
    'Longitude': [67.01],
    'Month': [4],
    'City': [df_sample['City'].iloc[0]]
})

prediction = model.predict(new_data)
print("Prediction:", prediction)

Prediction: [106]


In [20]:
crime_weight = {
    'Murder': 5,
    'Rape': 5,
    'Robbery': 4,
    'Assault': 3,
    'Theft': 2,
    'Burglary': 2,
    'Other': 1
}

df['Crime_Weight'] = df['Crime_Type'].map(crime_weight).fillna(1)

area_risk = df.groupby(['Latitude', 'Longitude']).agg({
    'Crime_Weight': 'sum'
}).reset_index()

def classify_risk(score):
    if score > 50:
        return "High Risk"
    elif score > 20:
        return "Medium Risk"
    else:
        return "Low Risk"

area_risk['Risk_Level'] = area_risk['Crime_Weight'].apply(classify_risk)

df = pd.merge(df, area_risk, on=['Latitude', 'Longitude'], how='left')

In [21]:
df.to_csv(r"C:\Users\yadav\OneDrive\사진\Desktop\CrimeProject\combined_crime_data.csv", index=False)

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import joblib

# Use only required columns
X = df[['Latitude', 'Longitude', 'Month']]
y = df['Crime_Type']

# Reduce data size (IMPORTANT)
df_small = df.sample(10000)   # take only 10k rows

X = df_small[['Latitude', 'Longitude', 'Month']]
y = df_small['Crime_Type']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ✅ LIGHTWEIGHT MODEL
model = DecisionTreeClassifier(max_depth=10)

# Train
# SAVE MODEL (IMPORTANT)
joblib.dump(model, r"C:\Users\yadav\OneDrive\사진\Desktop\CrimeProject\crime_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [23]:
import joblib
joblib.dump(model, "crime_model.pkl")

['crime_model.pkl']

In [24]:
pd.read_csv("combined_crime_data.csv", nrows=50000)

,Date,Latitude,Longitude,Crime_Type,City,Month
0,2024-12-15,24.859200,66.999800,Gang Violence,Karachi,12.0
1,2020-12-03,24.859200,66.999800,Murder,Karachi,12.0
2,2025-01-24,24.859200,66.999800,Murder,Karachi,1.0
3,2023-02-18,24.859200,66.999800,Gang Violence,Karachi,2.0
4,2025-04-13,24.859200,66.999800,Gang Violence,Karachi,4.0
...,...,...,...,...,...,...
49995,2021-01-24,24.864664,66.991471,Political Violence,Karachi,1.0
49996,2025-05-11,24.864664,66.991471,Industrial Hazards,Karachi,5.0
49997,2021-07-15,24.864664,66.991471,Gang Activity,Karachi,7.0
49998,2022-08-03,24.864664,66.991471,Murder,Karachi,8.0


In [25]:
def load_data():
    df = pd.read_csv(
        "combined_crime_data.csv",
        nrows=50000,
        low_memory=True
    )
    return df

In [26]:
def load_data():
    df = pd.read_csv("combined_crime_data.csv", nrows=50000)

    # Reduce memory
    for col in df.select_dtypes(include=['float64']):
        df[col] = df[col].astype('float32')

    for col in df.select_dtypes(include=['int64']):
        df[col] = df[col].astype('int32')

    return df